# Gold Setup Step 6 — Classify with Embeddings + LLM
Classifies unclassified products using centroid similarity with confidence-based routing:

| Similarity | Action |
|---|---|
| ≥ 0.85 | Auto-assign (`metodo='embedding_auto'`) |
| 0.60 – 0.85 | LLM confirms → review queue |
| < 0.60 | LLM proposes new category → review queue |

**Prerequisites:** run `05_build_category_embeddings.py.ipynb` first.

**Outputs:**
- `workspace.superapp.gold_productos_categorias` (auto-assigned rows appended)
- `workspace.superapp.gold_review_queue` (uncertain rows for notebook 07)

In [ ]:
%pip install sentence-transformers --quiet
dbutils.library.restartPython()

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import json
import requests
from datetime import datetime
from pyspark.sql.functions import col

CONFIDENCE_AUTO = 0.85
CONFIDENCE_LLM  = 0.60
LLM_ENDPOINT    = 'databricks-meta-llama-3-3-70b-instruct'
BATCH_SIZE      = 500

print('Config loaded.')
print(f'  Auto-assign threshold : {CONFIDENCE_AUTO}')
print(f'  LLM confirm threshold : {CONFIDENCE_LLM}')

In [ ]:
# Load centroids built by notebook 05.
centroids_pd = spark.table('workspace.superapp.gold_category_centroids').toPandas()
centroids_pd['centroid'] = centroids_pd['centroid_json'].apply(json.loads).apply(np.array)

centroid_matrix = np.array(centroids_pd['centroid'].tolist())  # (n_cats, dim)
category_names  = centroids_pd['nombre'].tolist()
cat_id_map      = dict(zip(centroids_pd['nombre'], centroids_pd['id_categoria']))

print(f'Centroids loaded: {len(centroids_pd)} categories')
print(f'Centroid matrix : {centroid_matrix.shape}')

In [ ]:
# Products not yet classified (not in mapping table, or mapped to NULL).
unclassified = spark.sql("""
    SELECT DISTINCT sp.id_producto, sp.producto
    FROM workspace.superapp.silver_prices sp
    LEFT JOIN workspace.superapp.gold_productos_categorias pc
        ON sp.id_producto = pc.id_producto
    WHERE (pc.id_producto IS NULL OR pc.id_categoria IS NULL)
      AND sp.producto IS NOT NULL
""").toPandas()

print(f'Unclassified products to process: {len(unclassified):,}')

In [ ]:
# Pre-load up to 5 example names per category — gives the LLM context (memory).
examples_pd = spark.sql("""
    SELECT gc.nombre, sp.producto
    FROM workspace.superapp.gold_productos_categorias pc
    JOIN workspace.superapp.gold_categorias gc ON pc.id_categoria = gc.id_categoria
    JOIN workspace.superapp.silver_prices sp    ON pc.id_producto = sp.id_producto
    WHERE gc.nombre != 'sin_clasificar' AND sp.producto IS NOT NULL
""").toPandas()

cat_examples = {}
for cat, grp in examples_pd.groupby('nombre'):
    cat_examples[cat] = grp['producto'].drop_duplicates().head(5).tolist()

print(f'Examples loaded for {len(cat_examples)} categories.')

In [ ]:
def cosine_sim_batch(embeddings, centroid_matrix):
    norms  = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normed = embeddings / (norms + 1e-10)
    return normed @ centroid_matrix.T  # (n_products, n_cats)


def get_token():
    return dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()


def ask_llm(product_name, top_category, top_score, examples, all_categories):
    workspace_url = spark.conf.get('spark.databricks.workspaceUrl')
    all_cats_str  = ', '.join(all_categories[:60])

    if top_score >= CONFIDENCE_LLM:
        examples_str = '\n'.join(f'  - {e}' for e in examples)
        prompt = (
            f'Producto de supermercado argentino: "{product_name}"\n\n'
            f'Categoria candidata: "{top_category}" (similitud={top_score:.2f})\n'
            f'Ejemplos de esa categoria:\n{examples_str}\n\n'
            f'Pertenece este producto a "{top_category}"?\n'
            f'Responde EXACTAMENTE: "SI" si pertenece, '
            f'"NO: <categoria>" con la categoria correcta (existente o nueva en snake_case espanol).\n\n'
            f'Categorias existentes: {all_cats_str}\n\nRespuesta:'
        )
    else:
        prompt = (
            f'Producto de supermercado argentino: "{product_name}"\n\n'
            f'No encontre categoria similar (mejor: "{top_category}", similitud={top_score:.2f})\n\n'
            f'Categorias existentes: {all_cats_str}\n\n'
            f'A que categoria pertenece? Responde SOLO con el nombre '
            f'(existente o nuevo en snake_case espanol).\n\nRespuesta:'
        )
    try:
        resp = requests.post(
            f'https://{workspace_url}/serving-endpoints/{LLM_ENDPOINT}/invocations',
            headers={'Authorization': f'Bearer {get_token()}', 'Content-Type': 'application/json'},
            json={'messages': [{'role': 'user', 'content': prompt}], 'max_tokens': 30, 'temperature': 0.1},
            timeout=30
        )
        if resp.status_code == 200:
            return resp.json()['choices'][0]['message']['content'].strip().lower()
    except Exception as e:
        print(f'  LLM error: {str(e)[:80]}')
    return None


def parse_llm_answer(answer, top_category, all_categories):
    if answer is None:
        return top_category, False
    if answer == 'si':
        return top_category, False
    suggested = answer[3:].strip() if answer.startswith('no:') else answer
    if suggested in all_categories:
        return suggested, False
    for cat in all_categories:
        if cat in suggested or suggested in cat:
            return cat, False
    return suggested, True  # new category


print('Helper functions defined.')

In [ ]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

auto_assigned = []
review_queue  = []
total         = len(unclassified)

print(f'Classifying {total:,} products in batches of {BATCH_SIZE}...\n')

for batch_start in range(0, total, BATCH_SIZE):
    batch = unclassified.iloc[batch_start : batch_start + BATCH_SIZE]
    embs  = model.encode(batch['producto'].tolist(), batch_size=128, show_progress_bar=False)
    sims  = cosine_sim_batch(embs, centroid_matrix)
    top_idx    = sims.argmax(axis=1)
    top_scores = sims.max(axis=1)

    for i, (_, row) in enumerate(batch.iterrows()):
        score    = float(top_scores[i])
        cat_name = category_names[top_idx[i]]
        id_cat   = cat_id_map.get(cat_name)

        if score >= CONFIDENCE_AUTO:
            auto_assigned.append({
                'id_producto': row['id_producto'], 'id_categoria': id_cat,
                'id_subcategoria': None, 'metodo': 'embedding_auto',
                'confianza': round(score, 4), 'fecha_asignacion': datetime.now(),
                'usuario_asignacion': 'ml_sistema',
                'notas': f'centroid_sim={score:.3f}|cat={cat_name}',
            })
        else:
            examples = cat_examples.get(cat_name, [])
            answer   = ask_llm(row['producto'], cat_name, score, examples, category_names)
            resolved, is_new = parse_llm_answer(answer, cat_name, category_names)
            review_queue.append({
                'id_producto': row['id_producto'], 'producto': row['producto'],
                'top_categoria': cat_name, 'similitud': round(score, 4),
                'llm_sugerencia': resolved, 'es_categoria_nueva': is_new,
                'razon': 'categoria_nueva' if is_new else 'confirmacion_llm',
                'estado': 'pendiente', 'fecha_creacion': datetime.now(),
            })

    pct = min(100, (batch_start + len(batch)) * 100 // total)
    print(f'  [{pct:3d}%]  auto={len(auto_assigned):,}  queue={len(review_queue):,}')

print(f'\n=== DONE ===')
print(f'  Auto-assigned : {len(auto_assigned):,}')
print(f'  Review queue  : {len(review_queue):,}')

In [ ]:
if auto_assigned:
    (spark.createDataFrame(auto_assigned)
          .write.mode('append').option('mergeSchema', 'true')
          .saveAsTable('workspace.superapp.gold_productos_categorias'))
    print(f'Appended {len(auto_assigned):,} rows to gold_productos_categorias')

if review_queue:
    (spark.createDataFrame(review_queue)
          .write.mode('append').option('mergeSchema', 'true')
          .saveAsTable('workspace.superapp.gold_review_queue'))
    print(f'Appended {len(review_queue):,} rows to gold_review_queue')

In [ ]:
spark.sql("""
    SELECT razon, es_categoria_nueva, estado, COUNT(*) AS total
    FROM workspace.superapp.gold_review_queue
    GROUP BY razon, es_categoria_nueva, estado
    ORDER BY total DESC
""").show()